# Actividad 3 — Patrón Factory Method (Creacional)

**Estudiante:** Andrés Felipe Luna Camargo  
**Dominio:** Monitoreo y Gestión de Cuartos Fríos (Cadena de Frío)

---

## 1. Contexto del Problema

En las bodegas y plantas de refrigeración tenemos diferentes tipos de cuartos fríos según el producto que se guarde:
- **Frutas y verduras (2°C a 4°C):** Usan equipos de compresión mecánica estándar con gas refrigerante (R404A/R134a).
- **Vacunas y biológicos (-70°C):** Requieren ultracongelación rápida mediante inyección de nitrógeno líquido.
- **Bodegas grandes de carnes (-20°C):** Funcionan con plantas de amoníaco industrial por su capacidad.

Cada una de estas máquinas funciona de forma distinta internamente (presiones, válvulas, tipo de gas), pero el sistema central siempre necesita hacer lo mismo: poner a enfriar el cuarto y generar el reporte del ciclo.

### El problema:
Si el despachador de la planta instancia directamente las clases concretas con un `if tipo == 'mecanico' ... elif tipo == 'criogenico'`, cada vez que compremos una máquina nueva nos toca venir a modificar ese código. Esto rompe el principio de Abierto/Cerrado (OCP) y deja todo amarrado.

**Solución con Factory Method:**  
Dejamos una clase base (`GestorFrigorifico`) con el método general `ejecutar_ciclo_frio()`, y definimos un método abstracto `crear_unidad()` para que cada subclase (`GestorMecanico`, `GestorCriogenico`, `GestorAmoniaco`) decida qué máquina específica instanciar.


## 2. Código Sin Patrón (Forma Incorrecta)

Aquí el despachador tiene que saber cómo se crea y cómo se llama cada método de cada máquina usando `if/elif`. Si agregamos una máquina nueva, toca modificar esta clase.


In [ ]:
# Código sin patrón: creación manual con ifs en el cliente

class CompresorMecanicoDirecto:
    def __init__(self, gas: str = "R404A"):
        self.gas = gas

    def arrancar_compresor(self, volumen: float, temp_meta: float) -> str:
        kwh = round(volumen * 0.4, 2)
        return f"[Mecánico] Enfriando {volumen}m3 a {temp_meta}°C con gas {self.gas}. Consumo: {kwh} kWh"


class InyectorCriogenicoDirecto:
    def __init__(self, pureza_n2: float = 99.9):
        self.pureza_n2 = pureza_n2

    def abrir_valvula_n2(self, volumen: float, temp_meta: float) -> str:
        litros = round(volumen * 1.8, 2)
        return f"[Criogénico] Inyectando N2 para {temp_meta}°C en {volumen}m3. Gasto: {litros} L de N2"


class DespachadorSinPatron:
    def despachar(self, tipo: str, cuarto: str, volumen: float, temp_meta: float):
        print(f"--- Despacho sin patrón para: {cuarto} ---")

        # Problema: si entra otra tecnología toca meter otro elif aquí
        if tipo == "mecanico":
            unidad = CompresorMecanicoDirecto(gas="R404A")
            res = unidad.arrancar_compresor(volumen, temp_meta)
        elif tipo == "criogenico":
            unidad = InyectorCriogenicoDirecto(pureza_n2=99.9)
            res = unidad.abrir_valvula_n2(volumen, temp_meta)
        else:
            raise ValueError(f"Tipo no soportado: {tipo}")

        print(f"Resultado: {res}\n")


# Probamos el despachador acoplado
despachador = DespachadorSinPatron()
despachador.despachar("mecanico", "Cuarto-Frutas-1", volumen=80.0, temp_meta=3.0)
despachador.despachar("criogenico", "Ultracongelador-Vacunas", volumen=25.0, temp_meta=-70.0)


--- Despacho sin patrón para: Cuarto-Frutas-1 ---
Resultado: [Mecánico] Enfriando 80.0m3 a 3.0°C con gas R404A. Consumo: 32.0 kWh

--- Despacho sin patrón para: Ultracongelador-Vacunas ---
Resultado: [Criogénico] Inyectando N2 para -70.0°C en 25.0m3. Gasto: 45.0 L de N2



## 3. Código Con Patrón Factory Method (Forma Correcta)

Organizamos las clases siguiendo los roles del patrón:
- **`IUnidadFrigorifica` (Interfaz Producto):** Define los métodos que todas las unidades deben tener (`activar_enfriamiento`, `nombre_equipo`).
- **Productos Concretos:** `UnidadCompresionMecanica`, `UnidadInyeccionCriogenica`, `UnidadAmoniacoIndustrial`.
- **`GestorFrigorifico` (Creador Abstracto):** Define el método de fábrica `crear_unidad()` y contiene el método `ejecutar_ciclo_frio()` que trabaja con la interfaz sin conocer la clase concreta.
- **Creadores Concretos:** `GestorMecanico`, `GestorCriogenico`, `GestorAmoniaco` implementan `crear_unidad()` devolviendo su objeto correspondiente.


In [ ]:
from abc import ABC, abstractmethod

# 1. Interfaz común para los equipos de frío (Producto)
class IUnidadFrigorifica(ABC):
    @abstractmethod
    def activar_enfriamiento(self, volumen_m3: float, temp_objetivo: float) -> str:
        pass

    @abstractmethod
    def nombre_equipo(self) -> str:
        pass


# 2. Clases concretas de cada tipo de equipo (Productos concretos)
class UnidadCompresionMecanica(IUnidadFrigorifica):
    def __init__(self, gas: str = "R404A", presion_psi: float = 120.0):
        self.gas = gas
        self.presion_psi = presion_psi

    def activar_enfriamiento(self, volumen_m3: float, temp_objetivo: float) -> str:
        kwh = round(volumen_m3 * 0.42, 2)
        return f"Compresor ({self.gas}) a {self.presion_psi} PSI. Enfriando {volumen_m3}m³ a {temp_objetivo}°C. Consumo: {kwh} kWh."

    def nombre_equipo(self) -> str:
        return f"Compresor Mecánico ({self.gas})"


class UnidadInyeccionCriogenica(IUnidadFrigorifica):
    def __init__(self, pureza_n2: float = 99.9):
        self.pureza_n2 = pureza_n2

    def activar_enfriamiento(self, volumen_m3: float, temp_objetivo: float) -> str:
        litros = round(volumen_m3 * 1.75, 2)
        return f"Inyección de N2 líquido (Pureza {self.pureza_n2}%). Ultracongelación rápida a {temp_objetivo}°C. Uso: {litros} L."

    def nombre_equipo(self) -> str:
        return "Inyección Criogénica (N2)"


class UnidadAmoniacoIndustrial(IUnidadFrigorifica):
    def __init__(self, toneladas_refrig: float = 80.0):
        self.toneladas_refrig = toneladas_refrig

    def activar_enfriamiento(self, volumen_m3: float, temp_objetivo: float) -> str:
        flujo = round(volumen_m3 * 3.2, 1)
        return f"Planta central de NH3 ({self.toneladas_refrig} TR). Recirculando {flujo} kg/h para {volumen_m3}m³ a {temp_objetivo}°C."

    def nombre_equipo(self) -> str:
        return f"Planta Amoníaco Industrial ({self.toneladas_refrig} TR)"


# 3. Clase base creadora con el Factory Method
class GestorFrigorifico(ABC):

    # Este es el Factory Method que cada subclase debe implementar
    @abstractmethod
    def crear_unidad(self) -> IUnidadFrigorifica:
        pass

    # Lógica común que usa el producto creado sin saber cuál es
    def ejecutar_ciclo_frio(self, cuarto_id: str, volumen_m3: float, temp_objetivo: float) -> str:
        # 1. La subclase crea la unidad
        unidad = self.crear_unidad()

        # 2. Usamos la unidad de forma polimórfica
        detalle = unidad.activar_enfriamiento(volumen_m3, temp_objetivo)

        reporte = (
            f"=== Ciclo de Enfriamiento ===\n"
            f"Cuarto: {cuarto_id}\n"
            f"Equipo: {unidad.nombre_equipo()}\n"
            f"Detalle: {detalle}\n"
        )
        return reporte


# 4. Creadores concretos: cada uno sabe cómo armar su unidad
class GestorMecanico(GestorFrigorifico):
    def __init__(self, gas: str = "R134a", presion: float = 115.0):
        self.gas = gas
        self.presion = presion

    def crear_unidad(self) -> IUnidadFrigorifica:
        return UnidadCompresionMecanica(gas=self.gas, presion_psi=self.presion)


class GestorCriogenico(GestorFrigorifico):
    def __init__(self, pureza: float = 99.95):
        self.pureza = pureza

    def crear_unidad(self) -> IUnidadFrigorifica:
        return UnidadInyeccionCriogenica(pureza_n2=self.pureza)


class GestorAmoniaco(GestorFrigorifico):
    def __init__(self, tr: float = 100.0):
        self.tr = tr

    def crear_unidad(self) -> IUnidadFrigorifica:
        return UnidadAmoniacoIndustrial(toneladas_refrig=self.tr)


# 5. Código cliente: solo trabaja con GestorFrigorifico
def despachar_cuarto(gestor: GestorFrigorifico, cuarto_id: str, volumen: float, temp_meta: float):
    print(gestor.ejecutar_ciclo_frio(cuarto_id, volumen, temp_meta))


# Pruebas con diferentes cuartos y tecnologías
print("--- 1. Cuarto de Manzanas ---")
despachar_cuarto(GestorMecanico(gas="R134a"), "Cuarto-Manzanas", volumen=60.0, temp_meta=3.0)

print("--- 2. Cuarto de Vacunas ---")
despachar_cuarto(GestorCriogenico(pureza=99.99), "Cuarto-Vacunas-Covid", volumen=20.0, temp_meta=-70.0)

print("--- 3. Bodega de Carnes ---")
despachar_cuarto(GestorAmoniaco(tr=150.0), "Bodega-Carnes-Principal", volumen=350.0, temp_meta=-22.0)


--- 1. Cuarto de Manzanas ---
=== Ciclo de Enfriamiento ===
Cuarto: Cuarto-Manzanas
Equipo: Compresor Mecánico (R134a)
Detalle: Compresor (R134a) a 115.0 PSI. Enfriando 60.0m³ a 3.0°C. Consumo: 25.2 kWh.

--- 2. Cuarto de Vacunas ---
=== Ciclo de Enfriamiento ===
Cuarto: Cuarto-Vacunas-Covid
Equipo: Inyección Criogénica (N2)
Detalle: Inyección de N2 líquido (Pureza 99.99%). Ultracongelación rápida a -70.0°C. Uso: 35.0 L.

--- 3. Bodega de Carnes ---
=== Ciclo de Enfriamiento ===
Cuarto: Bodega-Carnes-Principal
Equipo: Planta Amoníaco Industrial (150.0 TR)
Detalle: Planta central de NH3 (150.0 TR). Recirculando 1120.0 kg/h para 350.0m³ a -22.0°C.



## 4. Diagrama UML

```plantuml
@startuml
skinparam classAttributeIconSize 0

abstract class GestorFrigorifico {
    + {abstract} crear_unidad() : IUnidadFrigorifica
    + ejecutar_ciclo_frio(cuarto_id, volumen_m3, temp_objetivo) : str
}

class GestorMecanico {
    - gas: str
    - presion: float
    + crear_unidad() : IUnidadFrigorifica
}

class GestorCriogenico {
    - pureza: float
    + crear_unidad() : IUnidadFrigorifica
}

class GestorAmoniaco {
    - tr: float
    + crear_unidad() : IUnidadFrigorifica
}

GestorFrigorifico <|-- GestorMecanico
GestorFrigorifico <|-- GestorCriogenico
GestorFrigorifico <|-- GestorAmoniaco
GestorFrigorifico ..> IUnidadFrigorifica

interface IUnidadFrigorifica {
    + {abstract} activar_enfriamiento(volumen_m3, temp_objetivo) : str
    + {abstract} nombre_equipo() : str
}

class UnidadCompresionMecanica {
    + activar_enfriamiento(volumen_m3, temp_objetivo) : str
    + nombre_equipo() : str
}

class UnidadInyeccionCriogenica {
    + activar_enfriamiento(volumen_m3, temp_objetivo) : str
    + nombre_equipo() : str
}

class UnidadAmoniacoIndustrial {
    + activar_enfriamiento(volumen_m3, temp_objetivo) : str
    + nombre_equipo() : str
}

IUnidadFrigorifica <|.. UnidadCompresionMecanica
IUnidadFrigorifica <|.. UnidadInyeccionCriogenica
IUnidadFrigorifica <|.. UnidadAmoniacoIndustrial

GestorMecanico ..> UnidadCompresionMecanica : crea
GestorCriogenico ..> UnidadInyeccionCriogenica : crea
GestorAmoniaco ..> UnidadAmoniacoIndustrial : crea
@enduml
```


## 5. ¿Por qué elegí Factory Method y no otro patrón?

Revisando los otros patrones creacionales:

- **¿Por qué no Abstract Factory?**  
  Abstract Factory es cuando uno necesita crear familias completas de varios productos relacionados a la vez (por ejemplo: compresor + sensor + válvula de la misma marca). En nuestro caso solo necesitábamos instanciar un equipo de refrigeración a la vez, así que Abstract Factory hubiera sido meter complejidad sin necesidad.

- **¿Por qué no Builder?**  
  Builder se usa cuando armar un objeto toma muchos pasos o tiene demasiados parámetros opcionales. Aquí la máquina se crea de una sola vez con sus datos iniciales.

- **¿Por qué no Singleton?**  
  Singleton solo permite una sola instancia global. En una planta con varios cuartos fríos corriendo al tiempo necesitamos varios gestores independientes funcionando a la vez.

- **Conclusión:**  
  Factory Method era el patrón exacto porque nos permite dejar el flujo de trabajo (`ejecutar_ciclo_frio`) en una clase base y dejar que cada subclase decida qué máquina específica crear, cumpliendo con Open/Closed.
